In [1]:
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

## The schema

In [2]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
}

## Ask a question that needs current information

In [3]:
messages = [{"role": "user", "content": "What are the latest developments in Node.js LTS releases?"}]

response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    tools=[web_search_schema],
)
print(response.stop_reason)

end_turn


No loop is needed, because the server performs the searches and returns one finished response.

## Inspect every block

In [4]:
for block in response.content:
    print("TYPE:", block.type)
    if block.type == "text":
        print("  ", block.text[:200])
        for c in (getattr(block, "citations", None) or []):
            print("   cite ->", c.title, "|", c.url)
    elif block.type == "server_tool_use":
        print("   query:", block.input)
    elif block.type == "web_search_tool_result":
        for r in block.content:
            print("   result:", r.title, "|", r.url)
    print()

TYPE: server_tool_use
   query: {'query': 'Node.js LTS releases latest 2026'}

TYPE: web_search_tool_result
   result: Node.js 26 Released: What's New | https://www.inmotionhosting.com/support/news/nodejs-v26-released/
   result: Node.js release blog | https://nodejs.org/en/blog/release
   result: Node.js Version Support: EOL Dates and Latest Releases (July 2026) | https://www.herodevs.com/blog-posts/node-js-end-of-life-dates-you-should-be-aware-of
   result: github.com | https://github.com/nodejs-design-patterns-book/nodejsdesignpatterns.com/pull/37/files
   result: Question: Node.js 26 LTS Availability #9003 | https://github.com/nodejs/nodejs.org/discussions/9003
   result: Get Node.js LTS versions | https://npmjs.com/package/nodejs-lts-versions
   result: Releases · nodejs/node · GitHub | https://github.com/nodejs/node/releases
   result: Node.js Release and EOL Timeline - LTS Schedule, Active LTS and Maintenance, and End-of-Life Dates | https://hidekazu-konishi.com/entry/nodejs_rel

## Show the final answer with sources

In [5]:
answer = "".join(b.text for b in response.content if b.type == "text")
print(answer)

sources = {}
for b in response.content:
    if b.type == "text":
        for c in (getattr(b, "citations", None) or []):
            sources[c.url] = c.title
print("\nSources:")
for url, title in sources.items():
    print("-", title, url)

Based on the latest information, here are the key developments in Node.js LTS releases:

## Current LTS Status

The only fully supported lines today are Node.js 26 (Current), 24 (Active LTS), and 22 (Maintenance LTS).

Node.js 24 is the Active LTS line (EOL April 30, 2028) and Node.js 22 is in Maintenance LTS (EOL April 30, 2027). Node.js 26, released in May 2026, is the Current line and is scheduled to enter LTS in October 2026.

## Major Transition Coming

A significant change is coming to Node.js releases: Starting in October 2026, Node.js moves to one major release a year, version numbers line up with the calendar year, and every release becomes LTS. The old rule that odd-numbered releases never reach LTS ends after Node.js 26.

## Node.js 26 Features

Node.js 26 was released on May 5, 2026, with the Temporal API, V8 14.6, and breaking stream changes. Node.js 26 will enter LTS in October 2026. Active LTS receives bug fixes and security patches for 18 months, followed by a Maintenan

## Restrict to trusted domains

In [6]:
nih_schema = {**web_search_schema, "allowed_domains": ["nih.gov"]}

response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=[{"role": "user", "content": "What does research say about intermittent fasting?"}],
    tools=[nih_schema],
)
for b in response.content:
    if b.type == "web_search_tool_result":
        for r in b.content:
            print(r.url)

https://www.nia.nih.gov/news/research-intermittent-fasting-shows-health-benefits
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9298749/
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11566317/
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11247931/
https://clinicaltrials.gov/study/NCT04877314
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC12740946/
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC12899804/
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7551340/
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8469355/
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC12738305/
